# Predictions — base vs QLoRA-tuned

Inference only. Runs `src/generate_predictions.py` twice over the same 40 test
examples: once on the bare base model, once with the LoRA adapter from the
training kernel. Greedy decoding and a fixed seed, so both halves are comparable
and reproducible.

Inputs: the `sentry-sft-data-v1` dataset (test.jsonl + src/) and the output of
the `sentry-qlora-train` kernel (the adapter).

In [ ]:
# 1. Dependencies -- identical set to the training kernel that succeeded.
!pip install -q -U "transformers>=4.44,<6" "peft>=0.12" "bitsandbytes>=0.43" \
    "accelerate>=0.33" "datasets>=2.20" sentencepiece

In [ ]:
# 2. GPU check. The training run failed on a P100 (sm_60) because the image's
#    torch dropped sm_60; abort immediately rather than burn a slot.
import torch

assert torch.cuda.is_available(), "No GPU allocated."
name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
sm = f"sm_{cap[0]}{cap[1]}"
print("device :", name)
print("capability:", sm)
print("memory :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
assert cap >= (7, 0), (
    f"{name} is {sm}; this image's torch needs sm_70+. "
    "Set Accelerator to GPU T4 x2 in session settings."
)
print("GPU OK")

In [ ]:
# 3. Discover inputs. Kaggle's mount layout varies, so find files rather than
#    hardcode paths (a hardcoded path already cost one failed run).
from pathlib import Path
import os, shutil

ROOT = Path("/kaggle/input")
print("input tree (depth-limited):")
for p in sorted(ROOT.rglob("*")):
    if p.is_file() and p.suffix in (".jsonl", ".py", ".json", ".safetensors"):
        print("  ", p)

test_matches = sorted(ROOT.rglob("test.jsonl"))
assert test_matches, "test.jsonl not found under /kaggle/input"
TEST = test_matches[0]

src_matches = sorted(ROOT.rglob("generate_predictions.py"))
assert src_matches, "generate_predictions.py not found under /kaggle/input"
SRC = src_matches[0].parent

adapters = sorted(ROOT.rglob("adapter_config.json"))
assert adapters, "adapter_config.json not found -- is the training kernel attached as an input?"
# Prefer the top-level adapter dir over any checkpoint-* subdirectory.
top = [a for a in adapters if "checkpoint" not in str(a)]
ADAPTER = (top[0] if top else adapters[0]).parent

print("\ntest.jsonl :", TEST)
print("src dir    :", SRC)
print("adapter    :", ADAPTER)
print("adapter files:", sorted(p.name for p in ADAPTER.iterdir()))

In [ ]:
# 4. Stage into the working dir so `python -m src.generate_predictions` resolves.
WORK = Path("/kaggle/working")
shutil.copytree(SRC, WORK / "src", dirs_exist_ok=True)

DATA = WORK / "data" / "processed"
DATA.mkdir(parents=True, exist_ok=True)
shutil.copy(TEST, DATA / "test.jsonl")

ADAPTER_LOCAL = WORK / "adapter"
shutil.copytree(ADAPTER, ADAPTER_LOCAL, dirs_exist_ok=True)

os.chdir(WORK)
print("cwd:", Path.cwd())
print("test examples:", sum(1 for _ in open(DATA / "test.jsonl")))

In [ ]:
# 5. Training loss curve, straight from the adapter's own summary. This is the
#    verification of the training run that has not been done yet.
import json

summary_files = sorted(Path("/kaggle/input").rglob("training_summary.json"))
if summary_files:
    s = json.loads(summary_files[0].read_text())
    print("gpu             :", s.get("gpu"))
    print("steps           :", s.get("steps"))
    print("elapsed (min)   :", round(s.get("elapsed_seconds", 0) / 60, 1))
    print("final train loss:", s.get("final_train_loss"))
    print("final eval loss :", s.get("final_eval_loss"))
    print()
    print("full log_history:")
    for h in s.get("log_history", []):
        print("  ", json.dumps(h))
else:
    print("NO training_summary.json found in inputs -- cannot verify the loss curve here.")

In [ ]:
# 6. Baseline: bare Qwen2.5-3B-Instruct, no adapter.
!python -m src.generate_predictions --model base --load-4bit \
    --test data/processed/test.jsonl \
    --out /kaggle/working/preds_base.jsonl

In [ ]:
# 7. Tuned: same base model + the LoRA adapter, same seed, same decoding.
!python -m src.generate_predictions --model tuned --load-4bit \
    --adapter /kaggle/working/adapter \
    --test data/processed/test.jsonl \
    --out /kaggle/working/preds_tuned.jsonl

In [ ]:
# 8. Verify both files before declaring success, and clean non-output dirs.
import json, shutil
from pathlib import Path

for name in ("preds_base.jsonl", "preds_tuned.jsonl"):
    p = Path("/kaggle/working") / name
    assert p.exists(), f"MISSING {name}"
    rows = [json.loads(x) for x in p.read_text().splitlines() if x.strip()]
    assert len(rows) == 40, f"{name} has {len(rows)} predictions, expected 40"
    empty = sum(1 for r in rows if not r["prediction"].strip())
    print(f"{name}: {len(rows)} predictions, {p.stat().st_size/1024:.1f} KB, {empty} empty")

for d in ("src", "data", "adapter"):
    shutil.rmtree(Path("/kaggle/working") / d, ignore_errors=True)
print("\nfinal /kaggle/working:")
for p in sorted(Path("/kaggle/working").iterdir()):
    print("  ", p.name, p.stat().st_size)